# Practical Session 1: Markov Decision Processes and Finite-Horizon Dynamic Programming
##### M2 Artificial Intelligence (Université Paris-Saclay) - Reinforcement Learning, 2026-2027
Friday 20 November 2026

The statements are those of the exercise sheet of this session.

> 
> *The purpose of this work is to provide an introduction to the mathematical theory of multi-stage decision processes. Since these constitute a somewhat formidable set of terms we have coined the term “dynamic programming” to describe the subject matter.* […]
> *The problems we treat are programming problems, to use a terminology now popular. The adjective “dynamic,” however, indicates that we are interested in processes in which time plays a significant role, and in which the order of operations may be crucial. However, an essential feature of our approach will be the reinterpretation of many static processes as dynamic processes in which time can be artificially introduced.*
>
>  R. Bellman, *Dynamic Programming*, 1957, Preface, pages vii and xi.

The word “programming” refers to planning, in the sense of the operations research of the 1950s, and not to computer programming. This session follows Chapters 2 to 4 of the book of Puterman, *Markov Decision Processes: Discrete Stochastic Dynamic Programming* (1994), and its two running examples come from Sections 3.1 and 3.2 of that book.

**Remark 1 (Indexing of time).** Puterman numbers the decision epochs $1,\dots,N-1$ and calls $N$ the last epoch, at which only the terminal reward is received. This sheet numbers the decision epochs $0,\dots,N-1$ and receives the terminal reward at epoch $N$, so the horizon $N$ is the number of decisions taken, and a problem with horizon $N$ here has horizon $N+1$ in the book.

## Part I — Modelling a decision problem as a Markov decision process

To describe a sequential decision problem under uncertainty, one fixes what the decision maker observes, what it may do, how the system then moves, and what it receives.

**Definition 1 (Finite Markov decision process).** 
Let $\mathcal{S}$ be a nonempty finite set, the *state space*. For each state $s\in\mathcal{S}$, let $\mathcal{A}(s)$ be a nonempty finite set, the set of *actions admissible in* $s$, and let $\mathcal{A}=\bigcup_{s\in\mathcal{S}}\mathcal{A}(s)$. Write $\mathcal{K}=\{(s,a)\in\mathcal{S}\times\mathcal{A} : a\in\mathcal{A}(s)\}$ for the set of admissible state–action pairs. A *finite Markov decision process* is a quadruple $(\mathcal{S},\mathcal{A},P,r)$, where

- $P:\mathcal{K}\to\Delta(\mathcal{S})$, $(s,a)\mapsto P(\cdot\mid s,a)$, is the *transition probability*, with $\Delta(\mathcal{S})$ the set of probability distributions on $\mathcal{S}$,
- $r:\mathcal{K}\to\mathbb{R}$ is the *reward function*.

At each decision epoch $t$, the decision maker observes the state $S_t$, chooses an action $A_t\in\mathcal{A}(S_t)$, receives the reward $r(S_t,A_t)$, and the next state has the law

$$
\mathbb{P}(S_{t+1}=s'\mid S_t=s,A_t=a)=P(s'\mid s,a),\qquad (s,a)\in\mathcal{K},\ s'\in\mathcal{S} .
$$

A *finite-horizon* problem fixes in addition a horizon $N\in\mathbb{N}^{\ast}$, the decision epochs $t\in\{0,\dots,N-1\}$, and a *terminal reward* $r_{N}:\mathcal{S}\to\mathbb{R}$ received at epoch $N$.

**Example 1 (Two-state problem, Puterman 1994, Section 3.1).** The state space is $\{s_{1},s_{2}\}$. In state $s_{1}$ the decision maker chooses $a_{1,1}$ or $a_{1,2}$, and in state $s_{2}$ only $a_{2,1}$ is available. The dynamics and rewards are the following.

| State | Action | Next state | Reward |
|:---:|:---:|:---:|:---:|
| $s_{1}$ | $a_{1,1}$ | $s_{1}$ with probability $\tfrac12$, $s_{2}$ with probability $\tfrac12$ | $5$ |
| $s_{1}$ | $a_{1,2}$ | $s_{2}$ with probability $1$ | $10$ |
| $s_{2}$ | $a_{2,1}$ | $s_{2}$ with probability $1$ | $-1$ |

<div class="alert alert-warning">

#### Question 1: The two-state problem as a Markov decision process

Give the state space $\mathcal{S}$, the admissible action sets $\mathcal{A}(s)$, the reward function $r$ and the transition probability $P$ of the two-state problem.

</div>

**Example 2 (Single-product stochastic inventory control, Puterman 1994, Section 3.2).** 
At the beginning of each month, the manager of a warehouse observes the stock of a single product and decides how many units to order. The warehouse holds at most $M\in\mathbb{N}^{\ast}$ units, and the product is sold in whole units. An order is delivered at once. The demand $D_t$ of month $t$ takes values in $\mathbb{N}$, the demands of distinct months are independent with the common law $p_{j}=\mathbb{P}(D_t=j)$, $j\in\mathbb{N}$, and a demand that exceeds the stock is lost. Writing $s_t$ for the stock at the beginning of month $t$ and $a_t$ for the quantity ordered, the stock evolves as

$$
s_{t+1}=\left[s_t+a_t-D_t\right]^{+}.
$$

The economic data are, for a stock $u$ available after ordering: the expected revenue $F(u)$ of the month, the holding cost $h(u)$ of the month, and the ordering cost $O(a)$ of an order of $a$ units, with $O(0)=0$. The reward and the terminal reward are

$$
r(s,a)=F(s+a)-O(a)-h(s+a),\qquad
 r_{N}(s)=g(s),
$$

where $g(s)$ is the value of the stock left at the end of the horizon. To obtain a next stock $s'\ge1$ from a stock $s+a$ after ordering, the demand must equal $s+a-s'$, and the next stock is $0$ when the demand is at least $s+a$. With $q_{u}=\sum_{k\ge u}p_{k}=\mathbb{P}(D_t\ge u)$, this gives

$$
P(s'\mid s,a)=
\begin{cases}
0 & \text{if } s'>s+a,\\
p_{s+a-s'} & \text{if } 1\le s'\le s+a,\\
q_{s+a} & \text{if } s'=0 .
\end{cases}
$$

<div class="alert alert-warning">

#### Question 2: The inventory problem as a Markov decision process

Give the state space $\mathcal{S}$ and the admissible action sets $\mathcal{A}(s)$ of the inventory problem.

</div>

## Part II — Policies and value functions

A decision may depend on everything observed so far. Let $\mathscr{H}_0=\mathcal{S}$ and, for $t\in\mathbb{N}^{\ast}$, let $\mathscr{H}_t$ be the set of *histories* $h_t=(s_0,a_0,s_1,\dots,a_{t-1},s_t)$ with $a_k\in\mathcal{A}(s_k)$ for every $k\in\{0,\dots,t-1\}$. The random history up to epoch $t$ is written $H_t=(S_0,A_0,\dots,S_t)$.

**Definition 2 (Decision rule and policy).** A *decision rule* at epoch $t$ is a map $d_t:\mathscr{H}_t\to\Delta(\mathcal{A})$ such that, for every history $h_t\in\mathscr{H}_t$ with last state $s_t$, the distribution $d_t(h_t)$ gives probability $1$ to $\mathcal{A}(s_t)$. The decision rule is *deterministic* when each $d_t(h_t)$ is a Dirac mass, and it is then identified with a map $d_t:\mathscr{H}_t\to\mathcal{A}$ with $d_t(h_t)\in\mathcal{A}(s_t)$. A *policy* is a sequence $\pi=(d_0,\dots,d_{N-1})$ of decision rules. The policy is *Markovian* when each $d_t(h_t)$ depends on $h_t$ only through its last state $s_t$, so that $d_t$ is identified with a map defined on $\mathcal{S}$. A Markovian policy is *stationary* when $d_t=d$ for every $t$.

A policy $\pi$ and an initial state $s\in\mathcal{S}$ determine the law of the process $(S_0,A_0,S_1,\dots,S_{N})$, with $S_0=s$, $A_t$ drawn from $d_t(H_t)$ and $S_{t+1}$ drawn from $P(\cdot\mid S_t,A_t)$. This law is written $\mathbb{P}^{\pi}_{s}$, and its expectation $\mathbb{E}^{\pi}_{s}$.

<div class="alert alert-warning">

#### Question 3: Examples of policies

Let $N=2$. Give a deterministic Markovian policy $\pi=(d_0,d_1)$ and a randomised Markovian policy for the two-state problem.

</div>

<div class="alert alert-warning">

#### Question 4: History-dependent and Markovian policies in the two-state problem

In the two-state problem, compare the set of history-dependent policies with the set of Markovian policies.

</div>

<div class="alert alert-warning">

#### Question 5: Number of deterministic Markovian policies

Suppose that every state admits the same set of actions, $\mathcal{A}(s)=\mathcal{A}$ for every $s\in\mathcal{S}$. How many deterministic Markovian policies are there over a horizon $N$? How many of them are stationary?

</div>

<div class="alert alert-warning">

#### Question 6: A history-dependent policy

Add to the two-state problem an action $a_{1,3}$ in state $s_{1}$, with $P(s_{1}\mid s_{1},a_{1,3})=1$ and $r(s_{1},a_{1,3})=0$, so that $\mathcal{A}(s_{1})=\{a_{1,1},a_{1,2},a_{1,3}\}$. With $N=2$, give a deterministic policy that is history-dependent and not Markovian.

</div>

<div class="alert alert-warning">

#### Question 7: Comparing two policies

Let $N=2$, take no terminal reward, and let $R_t=r(S_t,A_t)$. Two policies $\pi$ and $\pi'$, started at the same state, satisfy

$$
\begin{gathered}
\mathbb{P}^{\pi}(R_0=i,R_1=j)=\tfrac14 \text{ for every } (i,j)\in\{0,1\}^2,\\
\mathbb{P}^{\pi'}(R_0=0,R_1=0)=\mathbb{P}^{\pi'}(R_0=1,R_1=1)=\tfrac12 .
\end{gathered}
$$

Compare the expected total rewards $\mathbb{E}^{\pi}[R_0+R_1]$ and $\mathbb{E}^{\pi'}[R_0+R_1]$, then the variances $\operatorname{Var}^{\pi}[R_0+R_1]$ and $\operatorname{Var}^{\pi'}[R_0+R_1]$. Which policy would you choose, and why?

</div>

**Definition 3 (Expected total reward).** The *expected total reward* of a policy $\pi$ started at $s\in\mathcal{S}$ is

$$
V^{\pi}_{N}(s)=\mathbb{E}^{\pi}_{s}\Big[\sum_{t=0}^{N-1}r(S_t,A_t)+r_{N}(S_{N})\Big].
$$

<div class="alert alert-warning">

#### Question 8: Expected total reward of a deterministic Markovian policy

Let $\pi=(d_0,\dots,d_{N-1})$ be a deterministic Markovian policy. Simplify the expression of $V^{\pi}_{N}(s)$.

</div>

### Finite-horizon policy evaluation

To compute $V^{\pi}_{N}$ without enumerating trajectories, the expected reward is evaluated backward in time, one epoch at a time. For a policy $\pi=(d_0,\dots,d_{N-1})$, an epoch $t\in\{0,\dots,N\}$ and a history $h_t\in\mathscr{H}_t$, let

$$
u^{\pi}_t(h_t)=\mathbb{E}^{\pi}\Big[\sum_{n=t}^{N-1}r(S_n,A_n)+r_{N}(S_{N})\,\Big|\,H_t=h_t\Big]
$$

be the expected reward collected from epoch $t$ on. In particular $u^{\pi}_0(s)=V^{\pi}_{N}(s)$.

**Finite-horizon policy evaluation algorithm, for a deterministic policy $\pi$.**

1. Set $t=N$ and $u^{\pi}_{N}(h_{N})=r_{N}(s_{N})$ for every $h_{N}=(h_{N-1},a_{N-1},s_{N})\in\mathscr{H}_{N}$.
2. If $t=0$, stop. Otherwise go to step 3.
3. Replace $t$ by $t-1$ and compute, for every $h_t=(h_{t-1},a_{t-1},s_t)\in\mathscr{H}_t$,
  
$$
u^{\pi}_t(h_t)=r\big(s_t,d_t(h_t)\big)+\sum_{s'\in\mathcal{S}}P\big(s'\mid s_t,d_t(h_t)\big)\,u^{\pi}_{t+1}\big(h_t,d_t(h_t),s'\big).
$$

4. Return to step 2.

The value from epoch $t$ on is the reward of the action $d_t(h_t)$ plus the expected value from epoch $t+1$ on: the computation over $N$ epochs reduces to $N$ one-epoch computations.

<div class="alert alert-warning">

#### Question 9: Cost of policy evaluation

Suppose that $\mathcal{A}(s)=\mathcal{A}$ for every $s\in\mathcal{S}$. Count the multiplications performed by the algorithm to compute $V^{\pi}_{N}(s)$ for every $s\in\mathcal{S}$, as a function of $N$, $|\mathcal{S}|$ and $|\mathcal{A}|$. Give the count when $\pi$ is Markovian.

</div>

<div class="alert alert-warning">

#### Question 10: Expressions of the policy evaluation

**(a)** Write the recursion of step 3 when $\pi$ is deterministic and Markovian.

**(b)** Write the recursion of step 3 with a conditional expectation.

</div>

## Part III — Optimality equations and the principle of optimality

The value of the problem is the best expected total reward achievable from each initial state.

**Definition 4 (Value and optimal policy).** The *value* of the problem is $V^{\ast}_{N}(s)=\sup_{\pi\in\Pi}V^{\pi}_{N}(s)$ for $s\in\mathcal{S}$, where $\Pi$ is the set of all policies, history-dependent and randomised ones included. A policy $\pi^{\ast}$ is *optimal* when $V^{\pi^{\ast}}_{N}(s)=V^{\ast}_{N}(s)$ for every $s\in\mathcal{S}$.

A family of functions $u_t:\mathscr{H}_t\to\mathbb{R}$, $t\in\{0,\dots,N\}$, satisfies the *optimality equations*, or *Bellman equations*, when

$$
u_t(h_t)=\max_{a\in\mathcal{A}(s_t)}\Big\{r(s_t,a)+\sum_{s'\in\mathcal{S}}P(s'\mid s_t,a)\,u_{t+1}(h_t,a,s')\Big\}
\tag{1}
$$

for every $t\in\{0,\dots,N-1\}$ and every $h_t\in\mathscr{H}_t$ with last state $s_t$, and $u_{N}(h_{N})=r_{N}(s_{N})$. The maximum exists because each $\mathcal{A}(s_t)$ is finite.

To relate the optimality equations to the value, Theorem 1 identifies their solution with the best expected reward from each epoch on: its proof bounds $u^{\pi}_t$ from above by $u_t$ for every policy, then constructs policies whose expected reward comes arbitrarily close to $u_t$.

**Theorem 1 (Optimality equations and the value; Puterman 1994, Theorem 4.3.2, page 84).** 
Let $(u_t)_{0\le t\le N}$ satisfy the optimality equations (1). Then

**(a)** $u_t(h_t)=\sup_{\pi\in\Pi}u^{\pi}_t(h_t)$ for every $t\in\{0,\dots,N\}$ and every $h_t\in\mathscr{H}_t$,

**(b)** $u_0(s)=V^{\ast}_{N}(s)$ for every $s\in\mathcal{S}$.

To obtain an optimal policy from the solution $(u^{\ast}_t)_{0\le t\le N}$ of the optimality equations, Theorem 2 selects at each epoch an action attaining the maximum in (1).

**Theorem 2 (Construction of an optimal policy; Puterman 1994, Theorem 4.3.3, page 86).** 
Let $(u^{\ast}_t)_{0\le t\le N}$ satisfy the optimality equations (1), and let $\pi^{\ast}=(d^{\ast}_0,\dots,d^{\ast}_{N-1})$ be a deterministic policy such that, for every $t\in\{0,\dots,N-1\}$ and every $h_t\in\mathscr{H}_t$ with last state $s_t$,

$$
d^{\ast}_t(h_t)\in\mathop{\mathrm{arg\,max}}\limits_{a\in\mathcal{A}(s_t)}\Big\{r(s_t,a)+\sum_{s'\in\mathcal{S}}P(s'\mid s_t,a)\,u^{\ast}_{t+1}(h_t,a,s')\Big\}.
$$

Then $u^{\pi^{\ast}}_t=u^{\ast}_t$ for every $t\in\{0,\dots,N\}$, and $\pi^{\ast}$ is optimal: $V^{\pi^{\ast}}_{N}=V^{\ast}_{N}$.

Puterman (page 87) presents Theorem 2 as a formal statement of the *principle of optimality*, which Bellman (1957, page 83, quoted there) phrased as follows: “An optimal policy has the property that whatever the initial state and initial decision are, the remaining decisions must constitute an optimal policy with regard to the state resulting from the first decision.”

To restrict the search to simple policies, Theorem 3 shows by backward induction that the solution of the optimality equations forgets the past: the terminal condition depends on the last state only, and each equation (1) preserves this property.

**Theorem 3 (Deterministic Markovian optimal policies; Puterman 1994, Theorem 4.4.2, page 89).** 
Let $(u^{\ast}_t)_{0\le t\le N}$ satisfy the optimality equations (1). Then

**(a)** for every $t\in\{0,\dots,N\}$, $u^{\ast}_t(h_t)$ depends on $h_t$ only through its last state $s_t$,

**(b)** there exists an optimal policy that is deterministic and Markovian.

Puterman states part (b) under the condition that the maximum in (1) is attained at every epoch and every state, a condition that holds here because every $\mathcal{A}(s)$ is finite.

<div class="alert alert-warning">

#### Question 11: Uniqueness of optimal policies

Is the optimal policy unique?

</div>

<div class="alert alert-warning">

#### Question 12: Optimality equations for Markovian policies

Write the optimality equations for functions of the current state only, and the corresponding optimal policy.

</div>

<div class="alert alert-warning">

#### Question 13: Cost of backward induction

Suppose that $\mathcal{A}(s)=\mathcal{A}$ for every $s\in\mathcal{S}$. Count the multiplications needed to compute $V^{\ast}_{N}(s)$ for every $s\in\mathcal{S}$ with the optimality equations for Markovian policies. Compare with the evaluation of every deterministic Markovian policy.

</div>

<div class="alert alert-warning">

#### Question 14: Solving the inventory problem

Consider the inventory problem of Example 2 with horizon $N=3$, so decisions at epochs $0$, $1$ and $2$, capacity $M=3$, and the following data:

- demand: $p_{0}=\tfrac14$, $p_{1}=\tfrac12$, $p_{2}=\tfrac14$, and $p_{j}=0$ for $j\ge3$,
- ordering cost: $O(0)=0$ and $O(a)=4+2a$ for $a\ge1$,
- holding cost $h(u)=u$, and terminal value $g(u)=0$,
- expected revenue: $F(0)=0$, $F(1)=6$, $F(2)=F(3)=8$.

**(a)** Compute the transition probabilities and the reward function.

**(b)** Apply backward induction to compute $V^{\ast}_{3}$ and an optimal policy. A short program may help: complete the code cells below.

</div>

In [ ]:
from fractions import Fraction

# Data of the question: horizon N = 3 (decisions at epochs 0, 1, 2), capacity M = 3.
decision_horizon = 3
inventory_capacity = 3
demand_probabilities = [Fraction(1, 4), Fraction(1, 2), Fraction(1, 4)]  # p_0, p_1, p_2, and p_j = 0 for j >= 3
expected_revenue = [0, 6, 8, 8]  # F(u) for u = 0, ..., M


def ordering_cost(order_quantity):
    """O(a): zero when nothing is ordered, 4 + 2a otherwise."""
    return 0 if order_quantity == 0 else 4 + 2 * order_quantity


def holding_cost(stock_after_order):
    """h(u) = u."""
    return stock_after_order


def terminal_reward(state):
    """r_N(s) = g(s) = 0."""
    return 0


def admissible_actions(state):
    """A(s) = {0, ..., M - s}."""
    return range(0, inventory_capacity - state + 1)


def transition_probabilities(state, action):
    """Return the list [P(s' | s, a) for s' = 0, ..., M]."""
    # TODO
    raise NotImplementedError


def reward(state, action):
    """Return r(s, a) = F(s + a) - O(a) - h(s + a)."""
    # TODO
    raise NotImplementedError


def backward_induction():
    """Return, for each epoch t = N-1, ..., 0, the values u*_t(s, a), u*_t(s) and the maximising actions."""
    # TODO
    raise NotImplementedError